MERGING DATASETS

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

def load_dataset(file_path):
    df = pd.read_csv(file_path, low_memory=False)
    return df
def load_dataset_tab_separation(file_path):
    df = pd.read_csv(file_path, low_memory=False, sep='\t')
    return df
def load_dataset_encoding(file_path):
    df = pd.read_csv(file_path, low_memory=False, encoding='latin1')
    return df

In [2]:
# IMDB imports
df_imdb_title_basics_c = load_dataset('../data/clean_data/cleaned_imdb_title_basics.csv') 
df_imdb_title_ratings_c = load_dataset('../data/imdb_datasets/title.ratings.csv') # no clean file
# OSCAR import
df_oscar_c = load_dataset('../data/clean_data/cleaned_oscar.csv') 
# BAFTA import
df_bafta_c = load_dataset('../data/clean_data/cleaned_bafta.csv')
# TMDB import
df_tmdb_c = load_dataset('../data/clean_data/cleaned_tmdb.csv')
# Box Office Mojo import
df_box_office_mojo_c = load_dataset_encoding('../data/clean_data/cleaned_box_office_mojo.csv')
# The Numbers import
df_the_numbers_c = load_dataset_encoding('../data/clean_data/cleaned_the_numbers.csv')
# Netflix imports
df_netflix_revenue_subs_spend_c = load_dataset('../data/clean_data/cleaned_netflix_revenue_subs_spend.csv')
df_netflix_engagement_c = load_dataset('../data/clean_data/cleaned_netflix_engagement.csv')

In [3]:
# df of interest:
df =  df_oscar_c

# Print top rows
print(df.head(10).to_string(), flush=True)

#print(len(df))

#print(df['category clean'].value_counts().to_string())

   Ceremony  Year   Class           CanonicalCategory                    Category      NomId               Film     FilmId             Name         Nominees NomineeIds Winner             Detail  MultifilmNomination
0        53  1980  Acting     ACTOR IN A LEADING ROLE     ACTOR IN A LEADING ROLE  an0054778        Raging Bull  tt0081398   Robert De Niro   Robert De Niro  nm0000134   True       Jake LaMotta                  NaN
1        53  1980  Acting     ACTOR IN A LEADING ROLE     ACTOR IN A LEADING ROLE  an0054779  The Great Santini  tt0079239    Robert Duvall    Robert Duvall  nm0000380    NaN       Bull Meechum                  NaN
2        53  1980  Acting     ACTOR IN A LEADING ROLE     ACTOR IN A LEADING ROLE  an0054780   The Elephant Man  tt0080678        John Hurt        John Hurt  nm0000457    NaN       John Merrick                  NaN
3        53  1980  Acting     ACTOR IN A LEADING ROLE     ACTOR IN A LEADING ROLE  an0054781            Tribute  tt0081656      Jack Lemmon 

In [ ]:
# df of interest:
df =  df_tmdb_c

# Don't do that !! Lowers the percentage !!! // df_tmdb_c = df[df['vote_count'] >= 1000]

# Print top rows
print(df.head(10).to_string(), flush=True)

len(df)

#print(df['Theatrical Distributor'].value_counts().to_string())

In [ ]:
import re

### TMDB and The Numbers

# Save original titles BEFORE cleaning
original_numbers_titles = df_the_numbers_c["Title"].copy()
original_tmdb_titles_2 = df_tmdb_c["title"].copy()

# Function to clean titles - remove all punctuation and whitespace
def clean_title(title):
    # Remove all non-word characters (punctuation and whitespace)
    return re.sub(r'[^\w]', '', str(title)).lower()

# Clean titles for merging
df_the_numbers_c["Title"] = df_the_numbers_c["Title"].apply(clean_title)
df_tmdb_c["title"] = df_tmdb_c["title"].apply(clean_title)

# Assigning the actual dataframes to df1 and df2
df1 = df_the_numbers_c
df2 = df_tmdb_c
# Define the matching columns explicitly
merge_column_df1 = "Title"  
merge_column_df2 = 'title' 

# Add prefix to df2 columns (except the join column)
df1_prefixed = df1.rename(
    columns=lambda c: f"numbers_{c}" if c != merge_column_df1 else c
)

# Perform an outer merge to keep all rows from both dataframes
merged_df = pd.merge(df1_prefixed, df2, left_on=merge_column_df1, right_on=merge_column_df2, how='left', indicator=True)

# Restore original titles in source dataframes after merge completes
df_the_numbers_c["Title"] = original_numbers_titles
df_tmdb_c["title"] = original_tmdb_titles_2

# Calculate number of matched rows
matched_rows = merged_df[merged_df['_merge'] == 'both'].shape[0]

# Total number of rows in both dataframes
total_df1_rows = df1.shape[0]
total_df2_rows = df2.shape[0]

# Calculate match percentage for each dataframe
match_percentage_df1 = (matched_rows / total_df1_rows) * 100
match_percentage_df2 = (matched_rows / total_df2_rows) * 100

# Display results
print(f"Total rows in df_1: {total_df1_rows}")
print(f"Total rows in df_2: {total_df2_rows}")
print(f"Matched rows: {matched_rows}")
print(f"Match percentage for df_1: {match_percentage_df1:.2f}%")
print(f"Match percentage for df_2: {match_percentage_df2:.2f}%")

# Display some of the rows that have matches
matched_rows = merged_df[merged_df['_merge'] == 'both']
#matched_rows_sample = matched_rows[['vote_average','vote_count','averageRating','numVotes']].head(2).to_string()
#print("\nSample of matched rows:")
#print(matched_rows_sample) 

#print("\nMatched rows:")
#print(matched_rows.head(20).to_string())

#Rows from df1 that did NOT match anything in df2
unmatched_df1 = merged_df[merged_df['_merge'] == 'left_only']

print("\nRows from df1 that did NOT match df2:")
print(unmatched_df1.head(20).to_string(index=False))   # show first 20


In [ ]:
import re

### TMDB and Box Office Mojo

# Save original titles BEFORE cleaning
original_box_titles = df_box_office_mojo_c["title"].copy()
original_tmdb_titles = df_tmdb_c["title"].copy()

# Function to clean titles - remove all punctuation and whitespace
def clean_title(title):
    # Remove all non-word characters (punctuation and whitespace)
    return re.sub(r'[^\w]', '', str(title)).lower()

# Clean titles for merging
df_box_office_mojo_c["title"] = df_box_office_mojo_c["title"].apply(clean_title)
df_tmdb_c["title"] = df_tmdb_c["title"].apply(clean_title)

# Assigning the actual dataframes to df1 and df2
df1 = df_box_office_mojo_c
df2 = df_tmdb_c
# Define the matching columns explicitly
merge_column_df1 = "title"  
merge_column_df2 = 'title' 

# Add prefix to df2 columns (except the join column)
df1_prefixed = df1.rename(
    columns=lambda c: f"mojo_{c}" if c != merge_column_df1 else c
)

# Perform an outer merge to keep all rows from both dataframes
merged_df = pd.merge(df1_prefixed, df2, left_on=merge_column_df1, right_on=merge_column_df2, how='left', indicator=True)

# Restore original titles in source dataframes after merge completes
df_box_office_mojo_c["title"] = original_box_titles
df_tmdb_c["title"] = original_tmdb_titles

# Calculate number of matched rows
matched_rows = merged_df[merged_df['_merge'] == 'both'].shape[0]

# Total number of rows in both dataframes
total_df1_rows = df1.shape[0]
total_df2_rows = df2.shape[0]

# Calculate match percentage for each dataframe
match_percentage_df1 = (matched_rows / total_df1_rows) * 100
match_percentage_df2 = (matched_rows / total_df2_rows) * 100

# Display results
print(f"Total rows in df_1: {total_df1_rows}")
print(f"Total rows in df_2: {total_df2_rows}")
print(f"Matched rows: {matched_rows}")
print(f"Match percentage for df_1: {match_percentage_df1:.2f}%")
print(f"Match percentage for df_2: {match_percentage_df2:.2f}%")

# Display some of the rows that have matches
matched_rows = merged_df[merged_df['_merge'] == 'both']
#matched_rows_sample = matched_rows[['vote_average','vote_count','averageRating','numVotes']].head(2).to_string()
#print("\nSample of matched rows:")
#print(matched_rows_sample) 

#print("\nMatched rows:")
#print(matched_rows.head(20).to_string())


#Rows from df1 that did NOT match anything in df2
unmatched_df1 = merged_df[merged_df['_merge'] == 'left_only']

print("\nRows from df1 that did NOT match df2:")
print(unmatched_df1.head(20).to_string(index=False))   # show first 20


In [ ]:
### TMDB and IMDB RATINGS

# TMBD Further Filtering
df_tmdb_c = df_tmdb_c[df_tmdb_c['vote_count'] >= 1000]  # Keep only movies with at least 1000 votes

#TEST
df_tmdb_c["imdb_id"] = df_tmdb_c["imdb_id"].str.strip().str.lower() 
df_imdb_title_ratings_c['tconst'] = df_imdb_title_ratings_c['tconst'].str.strip().str.lower()

# Assigning the actual dataframes to df1 and df2
df1 = df_tmdb_c 
df2 = df_imdb_title_ratings_c

# Define the matching columns explicitly
merge_column_df1 = "imdb_id"  
merge_column_df2 = 'tconst' 

# Perform an outer merge to keep all rows from both dataframes
merged_df = pd.merge(df1, df2, left_on=merge_column_df1, right_on=merge_column_df2, how='left', indicator=True)

# Calculate number of matched rows
matched_rows = merged_df[merged_df['_merge'] == 'both'].shape[0]

# Total number of rows in both dataframes
total_df1_rows = df1.shape[0]
total_df2_rows = df2.shape[0]

# Calculate match percentage for each dataframe
match_percentage_df1 = (matched_rows / total_df1_rows) * 100
match_percentage_df2 = (matched_rows / total_df2_rows) * 100
 
# Display results
print(f"Total rows in df_1: {total_df1_rows}")
print(f"Total rows in df_2: {total_df2_rows}")
print(f"Matched rows: {matched_rows}")
print(f"Match percentage for df_1: {match_percentage_df1:.2f}%")
print(f"Match percentage for df_2: {match_percentage_df2:.2f}%")

# Display some of the rows that have matches
matched_rows = merged_df[merged_df['_merge'] == 'both'].sort_values('vote_count', ascending=False)  # Sort by vote_count to show the most popular movies first
matched_rows_sample = matched_rows[['vote_average','vote_count','averageRating','numVotes']].head(2).to_string()
#print("\nSample of matched rows:")
#print(matched_rows_sample) 

print("\nMatched rows:")
print(matched_rows.head(10).to_string())

#Rows from df1 that did NOT match anything in df2
unmatched_df1 = merged_df[merged_df['_merge'] == 'left_only']

print("\nRows from df1 that did NOT match df2:")
print(unmatched_df1.head(20).to_string(index=False))   

In [ ]:
# imdb titles - filtering for what interests us

df= df_imdb_title_basics_c

df= df[df['titleType'] == 'movie']

df = df[df['startYear'] >= 1980]

print(len(df))

df_imdb_title_basics_c_filtered = df

### IMDB Names and Ratings

#TEST
df_imdb_title_basics_c_filtered["tconst"] = df_imdb_title_basics_c_filtered["tconst"].str.strip()
df_imdb_title_ratings_c["tconst"] = df_imdb_title_ratings_c["tconst"].str.strip()

# Assigning the actual dataframes to df1 and df2
df1 = df_imdb_title_basics_c_filtered  # Filter df1 to only include movies
df2 = df_imdb_title_ratings_c
# Define the matching columns explicitly
merge_column_df1 = 'tconst'  
merge_column_df2 = 'tconst' 

# Add prefix to df2 columns (except the join column)
#df1_prefixed = df1.rename(
    #columns=lambda c: f"numbers_{c}" if c != merge_column_df1 else c)

# Perform an outer merge to keep all rows from both dataframes
merged_df = pd.merge(df1, df2, left_on=merge_column_df1, right_on=merge_column_df2, how='left', indicator=True)

# Calculate number of matched rows
matched_rows = merged_df[merged_df['_merge'] == 'both'].shape[0]

# Total number of rows in both dataframes
total_df1_rows = df1.shape[0]
total_df2_rows = df2.shape[0]

# Calculate match percentage for each dataframe
match_percentage_df1 = (matched_rows / total_df1_rows) * 100
match_percentage_df2 = (matched_rows / total_df2_rows) * 100

# Display results
print(f"Total rows in df_1: {total_df1_rows}")
print(f"Total rows in df_2: {total_df2_rows}")
print(f"Matched rows: {matched_rows}")
print(f"Match percentage for df_1: {match_percentage_df1:.2f}%")
print(f"Match percentage for df_2: {match_percentage_df2:.2f}%")

# Display some of the rows that have matches
matched_rows = merged_df[merged_df['_merge'] == 'both']
#matched_rows_sample = matched_rows[['vote_average','vote_count','averageRating','numVotes']].head(2).to_string()
#print("\nSample of matched rows:")
#print(matched_rows_sample) 

#print("\nMatched rows:")
#print(matched_rows.head(20).to_string())

#Rows from df1 that did NOT match anything in df2
unmatched_df1 = merged_df[merged_df['_merge'] == 'left_only']

print("\nRows from df1 that did NOT match df2:")
print(unmatched_df1.head(20).to_string(index=False))   # show first 20

In [ ]:
### TMDB and OSCARS

# Assigning the actual dataframes to df1 and df2
df1 = df_oscar_c
df2 = df_tmdb_c
# Define the matching columns explicitly
merge_column_df1 = 'Film' #"FilmId"  
merge_column_df2 = 'title' #'imdb_id' 

#TEST
merge_column_df1 = df1[merge_column_df1].str.strip().str.lower()
merge_column_df2 = df2[merge_column_df2].str.strip().str.lower()

# Perform an outer merge to keep all rows from both dataframes
merged_df = pd.merge(df1, df2, left_on=merge_column_df1, right_on=merge_column_df2, how='left', indicator=True)

# Calculate number of matched rows
matched_rows = merged_df[merged_df['_merge'] == 'both'].shape[0]

# Total number of rows in both dataframes
total_df1_rows = df1.shape[0]
total_df2_rows = df2.shape[0]

# Calculate match percentage for each dataframe
match_percentage_df1 = (matched_rows / total_df1_rows) * 100
match_percentage_df2 = (matched_rows / total_df2_rows) * 100

# Display results
print(f"Total rows in df_1: {total_df1_rows}")
print(f"Total rows in df_2: {total_df2_rows}")
print(f"Matched rows: {matched_rows}")
print(f"Match percentage for df_1: {match_percentage_df1:.2f}%")
print(f"Match percentage for df_2: {match_percentage_df2:.2f}%")

# Display some of the rows that have matches
matched_rows = merged_df[merged_df['_merge'] == 'both']
#matched_rows_sample = matched_rows[['vote_average','vote_count','averageRating','numVotes']].head(2).to_string()
#print("\nSample of matched rows:")
#print(matched_rows_sample) 

#print("\nMatched rows:")
#print(matched_rows.head(10).to_string())

#Rows from df1 that did NOT match anything in df2
unmatched_df1 = merged_df[merged_df['_merge'] == 'left_only']

print("\nRows from df1 that did NOT match df2:")
print(unmatched_df1.head(20).to_string(index=False))  

In [ ]:
### TMDB and Bafta

#TEST
#df_bafta_c = df_bafta_c["nominee"].str.strip().str.lower() 
#df_tmdb_c["title"] = df_tmdb_c["title"].str.strip().str.lower()

# Assigning the actual dataframes to df1 and df2
df1 = df_bafta_c
df2 = df_tmdb_c
# Define the matching columns explicitly
merge_column_df1 = "nominee"  
merge_column_df2 = 'title' 

# Perform an outer merge to keep all rows from both dataframes
merged_df = pd.merge(df1, df2, left_on=merge_column_df1, right_on=merge_column_df2, how='left', indicator=True)

# Calculate number of matched rows
matched_rows = merged_df[merged_df['_merge'] == 'both'].shape[0]

# Total number of rows in both dataframes
total_df1_rows = df1.shape[0]
total_df2_rows = df2.shape[0]

# Calculate match percentage for each dataframe
match_percentage_df1 = (matched_rows / total_df1_rows) * 100
match_percentage_df2 = (matched_rows / total_df2_rows) * 100

# Display results
print(f"Total rows in df_1: {total_df1_rows}")
print(f"Total rows in df_2: {total_df2_rows}")
print(f"Matched rows: {matched_rows}")
print(f"Match percentage for df_1: {match_percentage_df1:.2f}%")
print(f"Match percentage for df_2: {match_percentage_df2:.2f}%")

# Display some of the rows that have matches
matched_rows = merged_df[merged_df['_merge'] == 'both']
#matched_rows_sample = matched_rows[['vote_average','vote_count','averageRating','numVotes']].head(2).to_string()
#print("\nSample of matched rows:")
#print(matched_rows_sample) 

print("\nMatched rows:")
print(matched_rows.head(10).to_string())

#Rows from df1 that did NOT match anything in df2
unmatched_df1 = merged_df[merged_df['_merge'] == 'left_only']

print("\nRows from df1 that did NOT match df2:")
print(unmatched_df1.head(50).to_string(index=False))  

In [ ]:

# Assign dfs for merging
df1 = df_imdb_title_ratings_c
df2 = df_imdb_title_basics_c
# Assign the ID matching columns
id1 = "tconst"
id2 = "tconst"

# Row counts before merge
df1_rows = len(df1)
df2_rows = len(df2)

print("DF1 rows before merge:", df1_rows)
print("DF2 rows before merge:", df2_rows)

# --- Merge DF1 and DF2 (left join) ---
df_merged = df1.merge(
    df2,
    how="left",
    left_on=id1,
    right_on=id2
)

# For DF1: match means DF2 id2 is not null
df1_matches = df_merged[id2].notna().sum()
df1_no_matches = df_merged[id2].isna().sum()

print("DF1 → DF2 matches:", df1_matches)
print("DF1 → DF2 no matches:", df1_no_matches)

# For DF2: match means DF2 rows whose id2 appears in DF1
df2_matches = df2[id2].isin(df1[id1]).sum()
df2_no_matches = df2_rows - df2_matches

print("DF2 rows matched in DF1:", df2_matches)
print("DF2 rows NOT matched in DF1:", df2_no_matches)

print("\n MATCHING % :\n")
print("DF2 match %:", df2_matches / df2_rows * 100)        # % of DF2 rows that appear in DF1
print("DF2 no match %:", df2_no_matches / df2_rows * 100)
print("DF1 match %:", df1_matches / df1_rows * 100)        # % of DF1 rows that found a match in DF2
print("DF1 no match %:", df1_no_matches / df1_rows * 100)

# ADD CODE TO INSPECT MATCHING COLUMNS


In [ ]:

# Row counts before merge
tmdb_rows = len(df_tmdb_c)
imdb_rows = len(df_imdb_title_basics_c)

print("TMDB rows before merge:", tmdb_rows)
print("IMDB rows before merge:", imdb_rows)


# --- Merge TMDB and IMDB title basics (left join) ---
df_tmdb_imdb_merged = df_tmdb_c.merge(
    df_imdb_title_basics_c,
    how="left",
    left_on="imdb_id",
    right_on="tconst"
)

#print("Merged dataset shape:", df_tmdb_imdb_merged.shape)

# For TMDB: match means IMDB tconst is not null
tmdb_matches = df_tmdb_imdb_merged['tconst'].notna().sum()
tmdb_no_matches = df_tmdb_imdb_merged['tconst'].isna().sum()

print("TMDB → IMDB matches:", tmdb_matches)
print("TMDB → IMDB no matches:", tmdb_no_matches)

imdb_matches = df_imdb_title_basics_c['tconst'].isin(df_tmdb_c['imdb_id']).sum()
imdb_no_matches = imdb_rows - imdb_matches

print("IMDB rows matched in TMDB:", imdb_matches)
print("IMDB rows NOT matched in TMDB:", imdb_no_matches)
print("\n MATCHING % :\n")
print("IMDB match %:", imdb_matches / imdb_rows * 100) #  Of all rows in the IMDB dataset, what percentage have a corresponding imdb_id in TMDB # IMDB rows that appear in TMDB
print("IMDB no match %:", imdb_no_matches / imdb_rows * 100)
print("TMDB match %:", tmdb_matches / tmdb_rows * 100) # Of all rows in the TMDB dataset, what percentage successfully found a match in IMDB # TMDB rows that appear in IMDB
print("TMDB no match %:", tmdb_no_matches / tmdb_rows * 100)



In [ ]:

df_tmdb_c['imdb_id'] = df_tmdb_c['imdb_id'].astype(str).str.strip()
df_imdb_title_basics_c['tconst'] = df_imdb_title_basics_c['tconst'].astype(str).str.strip()
